# 可选进阶实践：使用 RoboTwin 数据微调 LingBot-VLA-v2

本 notebook 保留经过路径验证的比赛参考 notebook 的 43-cell 技术覆盖，并把它组织成一条课程主线。notebook kernel 需要安装当前 `robo_genesis` package；GPU 重任务再通过 `/opt/robotwin-env/bin/python` 启动。必做路线是：preflight → base serving/evaluation → 4-GPU、100-step 动作专家 LoRA → DCP 转 Hugging Face checkpoint → merged serving/evaluation。全参数 SFT、official post-trained checkpoint、1000-episode benchmark 和 8-GPU profile 都继续作为可见的 opt-in 扩展保留。

当前固定 companion 能报告 aggregate `k/n`，但不能恢复每个实际 resolved seed 的 policy outcome。这不妨碍训练和部署。因此本 notebook 报告两个独立 run 的 aggregate 观测和 Wilson interval，不声称 resolved-seed matched improvement。本 notebook 中任何 GPU 实验都不能被写成 RoboGenesis 已经运行过的结果。


In [ ]:
from pathlib import Path
import ast, json, os, signal, subprocess, time, urllib.request

from robo_genesis.l14_workflow import (
    COMPANION_COMMIT, ROBOTWIN_COMMIT,
    archive_changed_files,
    audit_hf_checkpoint, audit_required_paths,
    build_aggregate_evidence_card, build_single_task_eval_command,
    command_text, l14_output_paths, parse_final_success_rate,
    parse_last_progress_rate,
    require_fresh_paths, run_command_logged, snapshot_files,
    validate_l14_config,
    write_json_artifact,
)

CONFIG = validate_l14_config({
    'run_name': 'l14-adjust-bottle-4gpu-001',  # Change for every independent run.
    'robotwin_root': '/RoboTwin',
    'runtime_root': '/workspace/runtime',
    'python': '/opt/robotwin-env/bin/python',
    'task_name': 'adjust_bottle',
    'task_config': 'demo_clean',
    'core_gpu_count': 4,
    'competition_gpu_count': 8,
    'lora_steps': 100,
    'eval_episodes': 10,
    'training_seed': 42,
})
ROBOTWIN = Path(CONFIG['robotwin_root'])
RUNTIME = Path(CONFIG['runtime_root'])
PYTHON = CONFIG['python']
PATHS = l14_output_paths(CONFIG)
TRAINING_ROOT = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
SOURCE_ROOT = TRAINING_ROOT / 'source/lingbot-vla-v2'
MODEL = TRAINING_ROOT / 'models/robbyant_lingbot-vla-v2-6b'
LINGBOT_VLA_COMMIT = '951475ae1b1d87553e7dc47c97b53a3d695c0d13'
PORT = 13400
EVAL_RESULT_ROOT = ROBOTWIN / 'eval_result'

os.environ.setdefault('AITER_TRITON_ONLY', '1')
os.environ.setdefault('FLASH_ATTENTION_TRITON_AMD_ENABLE', 'TRUE')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['TMPDIR'] = str(RUNTIME / 'tmp')
os.environ['PYTHONPATH'] = '/opt/aiter' + (
    ':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else ''
)
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print('Expected companion commit:', COMPANION_COMMIT)
print('Expected RoboTwin commit:', ROBOTWIN_COMMIT)


## 1. 环境、数据与身份 preflight

在分配模型显存前只运行一次 preflight。它检查固定仓库 revision、四张选定的 W7900、50 个数据集组成的 training manifest、三路相机、LoRA 配置、normalization 输入和 fresh output identity。任何失败都是停止条件，不能静默更换硬件或数据继续。


In [ ]:
required = {
    'objaverse index': ROBOTWIN / 'assets/objects/objaverse/list.json',
    'RoboTwin demos': ROBOTWIN / 'data/demo_clean',
    'LeRobot datasets': ROBOTWIN / 'data/lerobot',
    'training dataset list': ROBOTWIN / 'data/robotwin_demo_clean_joint_v30.txt',
    'base weight index': MODEL / 'model.safetensors.index.json',
    'LoRA training YAML': TRAINING_ROOT / 'training/lingbotvla_cli.yaml',
    'robot mapping YAML': SOURCE_ROOT / 'configs/robot_configs/robotwin.yaml',
    'normalization stats': SOURCE_ROOT / 'assets/norm_stats/robotwin.json',
}
input_audit = audit_required_paths(required)
require_fresh_paths([PATHS['run_root']])

companion_checkout = Path('/workspace/Robotwin-radeon-cloud')
companion_checkout_verified = (companion_checkout / '.git').exists()
companion_commit = subprocess.check_output(
    ['git', '-C', str(companion_checkout), 'rev-parse', 'HEAD'], text=True
).strip() if companion_checkout_verified else COMPANION_COMMIT
robotwin_commit = subprocess.check_output(
    ['git', '-C', str(ROBOTWIN), 'rev-parse', 'HEAD'], text=True
).strip()
lingbot_commit = subprocess.check_output(
    ['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
assert companion_commit == COMPANION_COMMIT, (companion_commit, COMPANION_COMMIT)
assert robotwin_commit == ROBOTWIN_COMMIT, (robotwin_commit, ROBOTWIN_COMMIT)
assert lingbot_commit == LINGBOT_VLA_COMMIT, (lingbot_commit, LINGBOT_VLA_COMMIT)

gpu_probe = subprocess.check_output([
    PYTHON, '-c',
    "import json, torch; print(json.dumps({'torch': torch.__version__, 'hip': torch.version.hip, 'available': torch.cuda.is_available(), 'count': torch.cuda.device_count(), 'names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))",
], text=True)
gpu = json.loads(gpu_probe.strip().splitlines()[-1])
assert gpu['hip'], 'PyTorch must report a HIP runtime'
assert gpu['available'] and gpu['count'] >= CONFIG['core_gpu_count'], gpu
selected_names = gpu['names'][:CONFIG['core_gpu_count']]
assert all('AMD Radeon PRO W7900' in name for name in selected_names), selected_names

dataset_entries = [
    line for line in required['training dataset list'].read_text().splitlines()
    if line.strip()
]
assert len(dataset_entries) == 50, len(dataset_entries)

manifest_rows = [entry.split(maxsplit=1) for entry in dataset_entries]
assert all(len(row) == 2 for row in manifest_rows), manifest_rows[:3]
dataset_paths = []
for _, raw_path in manifest_rows:
    dataset_path = Path(raw_path).expanduser()
    if not dataset_path.exists():
        try:
            relative = dataset_path.relative_to('/workspace/RoboTwin')
        except ValueError:
            pass
        else:
            relocated = ROBOTWIN / relative
            if relocated.exists():
                dataset_path = relocated
    assert dataset_path.is_dir(), f'Missing manifest dataset: {dataset_path}'
    dataset_paths.append(dataset_path.resolve())
assert len(set(dataset_paths)) == 50
sample_dataset = next(
    path for path in dataset_paths if path.name == 'adjust_bottle_joint_v30'
)
sample_info_path = sample_dataset / 'meta/info.json'
sample_info = json.loads(sample_info_path.read_text())
sample_features = sample_info['features']

import yaml
training_config = yaml.safe_load(required['LoRA training YAML'].read_text())
robot_config = yaml.safe_load(required['robot mapping YAML'].read_text())
assert training_config['data']['cameras'] == ['camera_top', 'camera_wrist_left', 'camera_wrist_right']
camera_mapping = {}
for item in robot_config['images']:
    target_key, mapping = next(iter(item.items()))
    camera_mapping[target_key.removeprefix('observation.images.')] = mapping['origin_keys']
assert camera_mapping == {
    'camera_top': 'observation.images.cam_high',
    'camera_wrist_left': 'observation.images.cam_left_wrist',
    'camera_wrist_right': 'observation.images.cam_right_wrist',
}
assert list(camera_mapping) == training_config['data']['cameras']
assert all(origin_key in sample_features for origin_key in camera_mapping.values())
raw_feature_shapes = {
    'observation.state': sample_features['observation.state']['shape'][-1],
    'action': sample_features['action']['shape'][-1],
}
assert raw_feature_shapes == {'observation.state': 14, 'action': 14}
def mapped_widths(section, expected_raw_key):
    widths = {}
    for item in robot_config[section]:
        target_key, mapping = next(iter(item.items()))
        origins = mapping['origin_keys']
        if isinstance(origins, str):
            assert origins in sample_features
            widths[target_key] = sample_features[origins]['shape'][-1]
            continue
        width = 0
        for origin in origins:
            raw_key, bounds = next(iter(origin.items()))
            assert raw_key == expected_raw_key
            assert 0 <= bounds['start'] < bounds['end'] <= raw_feature_shapes[raw_key]
            width += bounds['end'] - bounds['start']
        widths[target_key] = width
    return widths
mapped_state_widths = mapped_widths('states', 'observation.state')
mapped_action_widths = mapped_widths('actions', 'action')
assert mapped_state_widths == {
    'observation.state.arm.position': 12,
    'observation.state.effector.position': 2,
}
assert mapped_action_widths == {
    'action.arm.position': 12,
    'action.effector.position': 2,
}
semantic_groups = {}
for item in training_config['data']['joints']:
    semantic_groups.update(ast.literal_eval(item))
assert semantic_groups == {
    'arm.position': 14, 'end.position': 14, 'effector.position': 2,
}
for target_key, width in {**mapped_state_widths, **mapped_action_widths}.items():
    semantic_name = target_key.split('.', 2)[-1] if target_key.startswith('observation.') else target_key.removeprefix('action.')
    assert width <= semantic_groups[semantic_name]
assert training_config['train']['max_state_dim'] == 55
assert training_config['train']['max_action_dim'] == 55
norm_stats = json.loads(required['normalization stats'].read_text())
mapped_targets = {**mapped_state_widths, **mapped_action_widths}
assert set(mapped_targets).issubset(norm_stats['norm_stats'])
for target_key, width in mapped_targets.items():
    for stat_name in ('mean', 'std', 'q01', 'q99'):
        assert len(norm_stats['norm_stats'][target_key][stat_name]) == width
assert training_config['train']['use_lora'] is True
assert training_config['train']['lora_scope'] == 'action_expert'
assert training_config['train']['lora_rank'] == 8
assert training_config['train']['lora_alpha'] == 16
assert training_config['train']['seed'] == CONFIG['training_seed']

PATHS['logs'].mkdir(parents=True, exist_ok=False)
(RUNTIME / 'tmp').mkdir(parents=True, exist_ok=True)
run_manifest = {
    'schema_version': '1.0.0',
    'config': CONFIG,
    'companion_commit': companion_commit,
    'companion_checkout_verified': companion_checkout_verified,
    'robotwin_commit': robotwin_commit,
    'lingbot_vla_commit': lingbot_commit,
    'gpu': gpu,
    'dataset_count': len(dataset_entries),
    'sample_dataset': str(sample_dataset),
    'sample_features': sample_features,
    'robot_mapping_config': str(required['robot mapping YAML']),
    'camera_mapping': camera_mapping,
    'raw_feature_shapes': raw_feature_shapes,
    'mapped_state_widths': mapped_state_widths,
    'mapped_action_widths': mapped_action_widths,
    'normalized_targets': sorted(mapped_targets),
    'semantic_groups': semantic_groups,
    'model_capacity': {
        'max_state_dim': training_config['train']['max_state_dim'],
        'max_action_dim': training_config['train']['max_action_dim'],
    },
    'input_audit': input_audit,
    'resolved_seed_matching_available': False,
}
write_json_artifact(PATHS['manifest'], run_manifest)
EVAL_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Preflight passed:', PATHS['manifest'])


## 2. 启动 official base-model server

base checkpoint 同时是训练初始化点和部署参考。server 占用 GPU 0 与 13400 端口；下面的 cell 写入 fresh log，并等待 `/healthz` 成功后才进入评测。


In [ ]:
server_log = PATHS['logs'] / 'base_server.log'
require_fresh_paths([server_log])
server_command = [
    'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
    '0', str(PORT), str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('xb')
SERVER_PROCESS = subprocess.Popen(
    server_command, cwd=ROBOTWIN, stdout=server_log_handle,
    stderr=subprocess.STDOUT, start_new_session=True,
)
print('Base server command:', command_text(server_command))
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Base server exited; inspect {server_log}')
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
        print('Base server ready on port', PORT)
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Base server did not become ready; inspect {server_log}')


## 3. 闭环评测协议

必做的 base/merged 评测都使用 `adjust_bottle` / `demo_clean`、`expert_check=true`、`accept_expert_info_on_failure=true`、MPLib、EGL、10 episodes，并开启视频。固定 evaluator 在 setup 不稳定时可能推进内部 seed，但当前 log 只暴露 aggregate `k/n`。必须保留原始 log 与视频，不能根据 episode index 推断 matched outcome。


### 3.1 base checkpoint 是 pipeline reference

official base model 用于验证 serving、observation、action transport、simulation 和 scoring。较低成功数本身不代表部署链路损坏，10 episodes 也不是稳定的 policy benchmark。


In [ ]:
EVAL_VIDEO_LOG = True
BENCHMARK_VIDEO = False
EVAL_SEED_START = 100000
EVAL_MAX_COLLECTION_RUNS = 50
eval_env = os.environ.copy()
eval_env.update(
    ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib',
    PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1',
)
benchmark_env = eval_env.copy()
def collect_serial_episodes(
    condition, summary_log, attempts_json, video_root, environment,
):
    summary_log = Path(summary_log)
    attempts_json = Path(attempts_json)
    video_root = Path(video_root)
    attempts_dir = summary_log.parent / f'{summary_log.stem}_attempts'
    require_fresh_paths([summary_log, attempts_json, attempts_dir, video_root])
    attempts_dir.mkdir(parents=True)
    target_episodes = CONFIG['eval_episodes']
    completed = 0
    successes = 0
    seed_start = EVAL_SEED_START
    attempts = []
    video_artifacts = []
    while completed < target_episodes and len(attempts) < EVAL_MAX_COLLECTION_RUNS:
        requested_slots = target_episodes - completed
        command = build_single_task_eval_command(
            python=PYTHON, robotwin_root=ROBOTWIN,
            task_name=CONFIG['task_name'], task_config=CONFIG['task_config'],
            episodes=requested_slots, port=PORT, video=EVAL_VIDEO_LOG,
            batch=False, seed_start=seed_start,
        )
        video_before = snapshot_files(EVAL_RESULT_ROOT)
        attempt_index = len(attempts)
        raw_log = attempts_dir / (
            f'attempt_{attempt_index:02d}_seed_{seed_start}_slots_{requested_slots}.log'
        )
        print(f'{condition} evaluator command:', command_text(command))
        text = run_command_logged(
            command, cwd=ROBOTWIN, env=environment, log_path=raw_log,
        )
        progress = parse_last_progress_rate(text)
        attempt_successes, attempt_episodes = progress or (0, 0)
        assert attempt_episodes <= requested_slots
        assert completed + attempt_episodes <= target_episodes
        video_after = snapshot_files(EVAL_RESULT_ROOT)
        changed_videos = [
            relative for relative, metadata in video_after.items()
            if relative not in video_before or video_before[relative] != metadata
        ]
        if attempt_episodes:
            attempt_video_artifacts = archive_changed_files(
                EVAL_RESULT_ROOT, video_before,
                video_root / f'attempt_{attempt_index:02d}',
            )
            assert len(attempt_video_artifacts) == attempt_episodes
        else:
            assert not changed_videos
            attempt_video_artifacts = []
        video_artifacts.extend(attempt_video_artifacts)
        attempts.append({
            'attempt_index': attempt_index, 'seed_start': seed_start,
            'requested_slots': requested_slots,
            'completed_episodes': attempt_episodes,
            'successes': attempt_successes, 'raw_log': str(raw_log),
            'command': command_text(command),
            'video_artifacts': attempt_video_artifacts,
        })
        completed += attempt_episodes
        successes += attempt_successes
        seed_start += requested_slots
    if completed != target_episodes:
        raise RuntimeError(
            f'{condition} completed {completed}/{target_episodes} episodes after '
            f'{len(attempts)} serial evaluator runs'
        )
    assert len(video_artifacts) == target_episodes
    summary_text = '\n'.join(
        json.dumps(row, sort_keys=True) for row in attempts
    ) + (
        f'\nFinal success rate: {successes}/{completed} = '
        f'{successes / completed * 100:.1f}%\n'
    )
    summary_log.write_text(summary_text, encoding='utf-8')
    write_json_artifact(attempts_json, {
        'schema_version': '1.0.0', 'condition': condition,
        'target_episodes': target_episodes, 'completed_episodes': completed,
        'successes': successes, 'attempts': attempts,
    })
    return (
        parse_final_success_rate(
            summary_text, condition=condition, log_path=summary_log
        ),
        attempts,
        video_artifacts,
    )

representative_eval_command = build_single_task_eval_command(
    python=PYTHON, robotwin_root=ROBOTWIN,
    task_name=CONFIG['task_name'], task_config=CONFIG['task_config'],
    episodes=CONFIG['eval_episodes'], port=PORT, video=EVAL_VIDEO_LOG,
    batch=False, seed_start=EVAL_SEED_START,
)
print('Initial base evaluator command:', command_text(representative_eval_command))
base_eval_started = time.perf_counter()
BASE_RESULT, BASE_ATTEMPTS, BASE_VIDEO_ARTIFACTS = collect_serial_episodes(
    'base', PATHS['base_eval_log'], PATHS['run_root'] / 'base_attempts.json',
    PATHS['base_videos'], eval_env,
)
base_eval_elapsed = time.perf_counter() - base_eval_started
assert len(BASE_VIDEO_ARTIFACTS) == CONFIG['eval_episodes']
print('Base aggregate result:', BASE_RESULT)
print('Base evaluator runs:', BASE_ATTEMPTS)
print('Base video artifacts:', BASE_VIDEO_ARTIFACTS)
print(f'Base evaluation elapsed: {base_eval_elapsed / 60:.2f} min')


### 3.2 可选 official RoboTwin checkpoint

该 checkpoint 已经针对 RoboTwin 做过 post-training，可作为比赛参考，但不能作为 base→LoRA 教学实验的训练起点。完整代码路径继续保留，默认关闭，并使用独立的 log/result identity。


In [ ]:
RUN_OFFICIAL_ROBOTWIN_CHECKPOINT = False
ROBOTWIN_CHECKPOINT_ROOT = Path('/models/robotwin-persistent/models/robbyant_lingbot-vla-v2-6b-robotwin')
ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT
if not (ROBOTWIN_CHECKPOINT / 'model.safetensors.index.json').exists():
    ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT / 'checkpoints/global_step_50000/hf_ckpt'
ROBOTWIN_CHECKPOINT_CONFIG = ROBOTWIN_CHECKPOINT_ROOT / 'lingbotvla_cli.yaml'

if RUN_OFFICIAL_ROBOTWIN_CHECKPOINT:
    audit_hf_checkpoint(ROBOTWIN_CHECKPOINT)
    for process_name, handle_name in (
        ('SERVER_PROCESS', 'server_log_handle'),
        ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
    ):
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            process.wait(timeout=30)
            handle = globals().get(handle_name)
            if handle is not None and not handle.closed:
                handle.close()
    checkpoint_server_log = PATHS['logs'] / 'official_robotwin_server.log'
    require_fresh_paths([checkpoint_server_log])
    checkpoint_server_command = [
        'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
        '0', str(PORT), str(checkpoint_server_log), 'False', str(ROBOTWIN_CHECKPOINT),
    ]
    checkpoint_env = eval_env.copy()
    checkpoint_env['LINGBOTVLA_TRAINING_CONFIG'] = str(ROBOTWIN_CHECKPOINT_CONFIG)
    checkpoint_server_handle = checkpoint_server_log.open('xb')
    ROBOTWIN_CHECKPOINT_SERVER_PROCESS = subprocess.Popen(
        checkpoint_server_command, cwd=ROBOTWIN, env=checkpoint_env,
        stdout=checkpoint_server_handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(600):
        if ROBOTWIN_CHECKPOINT_SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f'Checkpoint server exited; inspect {checkpoint_server_log}')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
            break
        except Exception:
            time.sleep(2)
    official_eval_log = PATHS['logs'] / 'official_robotwin_evaluation.log'
    (
        OFFICIAL_ROBOTWIN_RESULT, OFFICIAL_ROBOTWIN_ATTEMPTS,
        OFFICIAL_VIDEO_ARTIFACTS,
    ) = collect_serial_episodes(
        'official_robotwin', official_eval_log,
        PATHS['run_root'] / 'official_robotwin_attempts.json',
        PATHS['official_videos'], checkpoint_env,
    )
    print(OFFICIAL_ROBOTWIN_RESULT)
else:
    print('SKIPPED: optional official RoboTwin checkpoint evaluation')


### 3.3 可选 official-base 100 tasks × 10 episodes

这是覆盖 clean 与 randomized tasks 的长 1000-episode benchmark。它会在每张选定 GPU 上启动一个 server，默认使用 8-GPU 比赛 profile，只有显式 opt-in 才运行。


In [ ]:
RUN_OFFICIAL_FULL_BENCHMARK = False
BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
BENCHMARK_EPISODES = 10
BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-official-both100x10-{BENCHMARK_GPU_COUNT}gpu"
BENCHMARK_RESUME = False
if RUN_OFFICIAL_FULL_BENCHMARK:
    if BENCHMARK_GPU_COUNT not in (4, 8):
        raise ValueError('BENCHMARK_GPU_COUNT must be 4 or 8')
    for process_name, handle_name in (
        ('SERVER_PROCESS', 'server_log_handle'),
        ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
    ):
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            process.wait(timeout=30)
            handle = globals().get(handle_name)
            if handle is not None and not handle.closed:
                handle.close()
    benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(BENCHMARK_GPU_COUNT), '--episodes', str(BENCHMARK_EPISODES),
        '--expert-check', '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    benchmark_run_root = RUNTIME / 'outputs' / BENCHMARK_RUN_NAME
    if BENCHMARK_RESUME:
        benchmark_command.append('--resume')
    else:
        require_fresh_paths([benchmark_run_root])
    subprocess.run(benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional official-base 100-task benchmark')


### 3.4 可选 official RoboTwin-checkpoint benchmark

这条对齐的长 benchmark 只改变 checkpoint identity，保留相同 scheduler 与 task set。它的 run name 和 outputs 必须与 official-base benchmark 隔离。


In [ ]:
RUN_ROBOTWIN_FULL_BENCHMARK = False
ROBOTWIN_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-robotwin-both100x10-{BENCHMARK_GPU_COUNT}gpu"
ROBOTWIN_BENCHMARK_RESUME = False
if RUN_ROBOTWIN_FULL_BENCHMARK:
    audit_hf_checkpoint(ROBOTWIN_CHECKPOINT)
    robotwin_benchmark_env = benchmark_env.copy()
    robotwin_benchmark_env['LINGBOTVLA_TRAINING_CONFIG'] = str(ROBOTWIN_CHECKPOINT_CONFIG)
    robotwin_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(BENCHMARK_GPU_COUNT), '--episodes', str(BENCHMARK_EPISODES),
        '--model-path', str(ROBOTWIN_CHECKPOINT), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', ROBOTWIN_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    robotwin_benchmark_root = RUNTIME / 'outputs' / ROBOTWIN_BENCHMARK_RUN_NAME
    if ROBOTWIN_BENCHMARK_RESUME:
        robotwin_benchmark_command.append('--resume')
    else:
        require_fresh_paths([robotwin_benchmark_root])
    subprocess.run(robotwin_benchmark_command, cwd=ROBOTWIN, env=robotwin_benchmark_env, check=True)
else:
    print('SKIPPED: optional official-RoboTwin 100-task benchmark')


## 4. 动作专家 LoRA、DCP merge 与 inference

必做训练 profile 使用四张 W7900 和 100 optimizer steps。它验证完整 pipeline，不证明 policy quality 普遍提升。8-GPU profile 继续保留给比赛规模的扩展实验。


### 4.1 训练动作专家 LoRA adapter

固定 YAML 在动作专家的 `q_proj`、`k_proj`、`v_proj` 和 `o_proj` 上应用 rank-8、alpha-16 LoRA。micro batch 1、四个 data-parallel shards、global batch 4 对应 gradient accumulation 1。FSDP2 对训练状态分片，seed 42 固定训练侧随机日程。


In [ ]:
for process_name, handle_name in (
    ('SERVER_PROCESS', 'server_log_handle'),
    ('ROBOTWIN_CHECKPOINT_SERVER_PROCESS', 'checkpoint_server_handle'),
):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)
        handle = globals().get(handle_name)
        if handle is not None and not handle.closed:
            handle.close()

GPU_COUNT = CONFIG['core_gpu_count']  # Change explicitly to competition_gpu_count for 8 GPUs.
TRAIN_STEPS = CONFIG['lora_steps']
LORA_GLOBAL_BATCH_SIZE = GPU_COUNT
LORA_MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = LORA_GLOBAL_BATCH_SIZE // (GPU_COUNT * LORA_MICRO_BATCH_SIZE)
training_yaml = TRAINING_ROOT / 'training/lingbotvla_cli.yaml'
training_output = PATHS['lora_output']
require_fresh_paths([training_output])
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.micro_batch_size', str(LORA_MICRO_BATCH_SIZE),
    '--train.gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--train.global_batch_size', str(LORA_GLOBAL_BATCH_SIZE),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.seed', str(CONFIG['training_seed']),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
print('LoRA command:', command_text(train_command))
training_started = time.perf_counter()
run_command_logged(
    train_command, cwd=SOURCE_ROOT, env=train_env,
    log_path=PATHS['logs'] / 'lora_training.log',
)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training elapsed: {training_elapsed / 60:.2f} min')


### 4.2 合并 LoRA DCP checkpoint

训练产物是分布式 DCP state，不是可以直接 serving 的模型。merge script 重建 adapter，核对 rank/alpha，把它应用到精确 base checkpoint，并写出 Hugging Face checkpoint。下一 cell 会审计 config、processor/tokenizer files、weight index 和所有引用 shard。


In [ ]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.is_dir(), f'Missing DCP checkpoint: {checkpoint}'
require_fresh_paths([MERGED])
merge_command = [
    PYTHON, str(TRAINING_ROOT / 'scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint), '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
print('LoRA merge command:', command_text(merge_command))
run_command_logged(
    merge_command, cwd=ROBOTWIN, env=os.environ.copy(),
    log_path=PATHS['logs'] / 'lora_merge.log',
)
MERGED_AUDIT = audit_hf_checkpoint(MERGED)
print(json.dumps(MERGED_AUDIT, indent=2))


### 4.3 启动 merged LoRA server

继续使用 13400 端口和相同 runtime flags。base 与 merged 两次评测之间，有意改变的变量应是 checkpoint identity。


In [ ]:
merged_log = PATHS['logs'] / 'merged_server.log'
require_fresh_paths([merged_log])
merged_server_command = [
    'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
    '0', str(PORT), str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('xb')
MERGED_SERVER_PROCESS = subprocess.Popen(
    merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle,
    stderr=subprocess.STDOUT, start_new_session=True,
)
print('Merged server command:', command_text(merged_server_command))
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited; inspect {merged_log}')
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
        print('Merged server ready on port', PORT)
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')


### 4.4 评测 merged LoRA checkpoint

重复同一条 10-episode 命令，并保留独立 log。由于当前 companion 不暴露最终逐 resolved-seed outcome，两个 `k/n` 只能作为独立小样本观测比较。即使 merged count 更高，也不能写成 matched improvement。


In [ ]:
print('Merged evaluation uses the same adaptive serial protocol as base.')
merged_eval_started = time.perf_counter()
MERGED_RESULT, MERGED_ATTEMPTS, MERGED_VIDEO_ARTIFACTS = collect_serial_episodes(
    'merged_lora', PATHS['merged_eval_log'],
    PATHS['run_root'] / 'merged_lora_attempts.json',
    PATHS['merged_videos'], eval_env,
)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
assert len(MERGED_VIDEO_ARTIFACTS) == CONFIG['eval_episodes']
print('Merged aggregate result:', MERGED_RESULT)
print('Merged evaluator runs:', MERGED_ATTEMPTS)
print('Merged video artifacts:', MERGED_VIDEO_ARTIFACTS)
print(f'Merged evaluation elapsed: {merged_eval_elapsed / 60:.2f} min')


### 4.5 可选 merged-LoRA 100 tasks × 10 episodes

长 benchmark 保留自己的 opt-in 与 run identity，不与必做的 10+10 smoke 路线混合。


In [ ]:
RUN_LORA_FULL_BENCHMARK = False
LORA_BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
LORA_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-lora-both100x10-{LORA_BENCHMARK_GPU_COUNT}gpu"
LORA_BENCHMARK_RESUME = False
if RUN_LORA_FULL_BENCHMARK:
    if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
        MERGED_SERVER_PROCESS.wait(timeout=30)
        merged_log_handle.close()
    lora_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(LORA_BENCHMARK_GPU_COUNT), '--episodes', '10',
        '--model-path', str(MERGED), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', LORA_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    lora_benchmark_root = RUNTIME / 'outputs' / LORA_BENCHMARK_RUN_NAME
    if LORA_BENCHMARK_RESUME:
        lora_benchmark_command.append('--resume')
    else:
        require_fresh_paths([lora_benchmark_root])
    subprocess.run(lora_benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional merged-LoRA 100-task benchmark')


### 4.6 停止 merged server 并写出范围恰当的 evidence card

evidence card 记录两个 aggregate count、Wilson interval、observed rate difference、固定身份，以及 matched outcomes 当前不可用这一事实。即使 merged count 更高，也必须保留这条限制。


In [ ]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    merged_log_handle.close()
    print('Merged model server stopped')

AGGREGATE_EVIDENCE = build_aggregate_evidence_card(
    BASE_RESULT,
    MERGED_RESULT,
    identities={
        'run_name': CONFIG['run_name'],
        'task_name': CONFIG['task_name'],
        'task_config': CONFIG['task_config'],
        'protocol': 'lingbot_vla_v2',
        'cli_seed': 0,
        'eval_episodes': CONFIG['eval_episodes'],
        'base_checkpoint': str(MODEL),
        'merged_lora_checkpoint': str(MERGED),
        'training_seed': CONFIG['training_seed'],
        'lora_steps': TRAIN_STEPS,
        'lora_rank': 8,
        'lora_alpha': 16,
    },
    video_artifacts={
        'base': BASE_VIDEO_ARTIFACTS,
        'merged_lora': MERGED_VIDEO_ARTIFACTS,
    },
)
write_json_artifact(PATHS['aggregate_evidence'], AGGREGATE_EVIDENCE)
print(json.dumps(AGGREGATE_EVIDENCE, indent=2))
print('IMPORTANT: matched outcomes are unavailable with the pinned companion.')
print('Do not claim that the observed aggregate difference is a resolved-seed-matched improvement.')


## 5. 可选全参数 SFT、转换与 inference

全参数 SFT 是一条独立训练路径：`use_lora=false` 会更新所有 trainable parameters，保留的默认配置还使用完整 depth/video teacher supervision。它在时间、显存和存储上都明显更昂贵，所以本节完整可见但默认关闭。启用后必须走完 training、DCP conversion、serving 与 evaluation，不能只看到 loss 就结束。


In [ ]:
subprocess.run(['df', '-h', '/', '/workspace'], check=True)
outputs_root = RUNTIME / 'outputs'
subprocess.run(['du', '-d1', '-h', str(outputs_root)], check=True)
cleanup_candidates = sorted(
    set(outputs_root.glob('lora_*'))
    | set(outputs_root.glob('reproduction_*'))
    | set(outputs_root.glob('full_sft_*'))
)
print('Dry-run cleanup candidates; this notebook does not delete them:')
for candidate in cleanup_candidates:
    print(' -', candidate)
print('Archive required checkpoints, logs, videos, and evidence before manual deletion.')


### 5.1 运行全参数 SFT

launcher 根据 global batch 256、GPU 数和 micro batch 推导 accumulation。四张 W7900 的参考配置是 micro batch 16、accumulation 4，并在 forward 后重新分片。启用前先检查上一 cell 的 dry-run 存储列表。


In [ ]:
RUN_FULL_SFT = False
FULL_SFT_GPU_COUNT = CONFIG['core_gpu_count']
FULL_SFT_STEPS = 10
FULL_SFT_SAVE_STEPS = FULL_SFT_STEPS
FULL_SFT_TEACHER_MODE = 'full'
FULL_SFT_TIMEOUT_SECONDS = 0
FULL_SFT_GLOBAL_BATCH_SIZE = 256
FULL_SFT_MICRO_BATCH_SIZE = 16
FULL_SFT_ENABLE_FULL_SHARD = True
FULL_SFT_OPTIMIZER = 'adamw'
FULL_SFT_OUTPUT = PATHS['full_sft_output']

if RUN_FULL_SFT:
    assert FULL_SFT_GPU_COUNT in (4, 8)
    assert FULL_SFT_TEACHER_MODE in ('full', 'current_depth', 'none')
    require_fresh_paths([FULL_SFT_OUTPUT])
    gpu_count = int(subprocess.check_output(
        [PYTHON, '-c', 'import torch; print(torch.cuda.device_count())'], text=True
    ).strip())
    assert gpu_count >= FULL_SFT_GPU_COUNT
    full_sft_env = os.environ.copy()
    full_sft_env.update({
        'GPU_COUNT': str(FULL_SFT_GPU_COUNT), 'MAX_STEPS': str(FULL_SFT_STEPS),
        'SAVE_STEPS': str(FULL_SFT_SAVE_STEPS), 'TEACHER_MODE': FULL_SFT_TEACHER_MODE,
        'TIMEOUT_SECONDS': str(FULL_SFT_TIMEOUT_SECONDS),
        'GLOBAL_BATCH_SIZE': str(FULL_SFT_GLOBAL_BATCH_SIZE),
        'MICRO_BATCH_SIZE': str(FULL_SFT_MICRO_BATCH_SIZE),
        'ENABLE_FULL_SHARD': str(FULL_SFT_ENABLE_FULL_SHARD).lower(),
        'OPTIMIZER': FULL_SFT_OPTIMIZER, 'OUTPUT_DIR': str(FULL_SFT_OUTPUT),
    })
    full_sft_command = [
        'bash', str(TRAINING_ROOT / 'training/train_full_sft.sh')
    ]
    print('Full-SFT command:', command_text(full_sft_command))
    run_command_logged(
        full_sft_command, cwd=ROBOTWIN, env=full_sft_env,
        log_path=PATHS['logs'] / 'full_sft_training.log',
    )
else:
    print('SKIPPED: full-parameter SFT is an opt-in high-cost extension')


### 5.2 固定 companion 的 full-teacher capacity 参考

以下是固定 companion 报告的比赛参考测量，不是 RoboGenesis 复验结果。steady time 是 optimizer steps 2 和 3 的均值；VRAM 是从加载、forward、backward 到 optimizer update 期间采样的最大值。

| 平台 | 每 GPU micro batch | Accumulation | Steady StepTime | 每 GPU Peak VRAM | 结果 |
|---|---:|---:|---:|---:|---|
| 4 × W7900 48 GiB | 1 | 64 | 245.580 s | 40.329–40.375 GiB | passed |
| 4 × W7900 48 GiB | 2 | 32 | 136.748 s | 41.337–41.349 GiB | passed |
| 4 × W7900 48 GiB | 4 | 16 | 76.010 s | 43.509–43.521 GiB | passed |
| 4 × W7900 48 GiB | 8 | 8 | 54.031 s | 47.729–47.761 GiB | passed；余量有限 |
| 4 × W7900 48 GiB | 16 | 4 | 41.392 s | 47.890–47.975 GiB | passed；接近满载 |
| 4 × W7900 48 GiB | 32 | 2 | — | 47.648–47.956 GiB | HIP OOM |
| 8 × W7900 48 GiB | 1 | 32 | 143.665 s | 27.520–27.558 GiB | computed |
| 8 × W7900 48 GiB | 2 | 16 | 80.319 s | 27.818–27.832 GiB | passed |
| 8 × W7900 48 GiB | 4 | 8 | 47.055 s | 29.498–29.523 GiB | passed |
| 8 × W7900 48 GiB | 8 | 4 | 27.108 s | 33.004–33.386 GiB | passed |
| 8 × W7900 48 GiB | 16 | 2 | 24.682 s | 41.344–41.356 GiB | passed |
| 8 × W7900 48 GiB | 32 | 1 | — | 47.878–47.947 GiB | HIP OOM |


### 5.3 把 full-SFT DCP 转换为 serving checkpoint

full SFT 没有 adapter 可供 merge。conversion 会 gather 分布式 model state，按固定实现过滤只用于训练的 teacher heads，并写出可以直接加载的 Hugging Face checkpoint。


In [ ]:
FULL_SFT_CHECKPOINT = FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}'
FULL_SFT_MODEL = FULL_SFT_OUTPUT / f'merged_checkpoint/global_step_{FULL_SFT_STEPS}/hf_ckpt'
if RUN_FULL_SFT:
    assert FULL_SFT_CHECKPOINT.is_dir(), f'Missing checkpoint: {FULL_SFT_CHECKPOINT}'
    require_fresh_paths([FULL_SFT_MODEL])
    full_sft_convert_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/convert_full_sft_dcp.py'),
        '--checkpoint', str(FULL_SFT_CHECKPOINT),
        '--training-output', str(FULL_SFT_OUTPUT),
        '--output', str(FULL_SFT_MODEL),
    ]
    run_command_logged(
        full_sft_convert_command, cwd=ROBOTWIN, env=os.environ.copy(),
        log_path=PATHS['logs'] / 'full_sft_conversion.log',
    )
    FULL_SFT_MODEL_AUDIT = audit_hf_checkpoint(FULL_SFT_MODEL)
    print(json.dumps(FULL_SFT_MODEL_AUDIT, indent=2))
else:
    print('SKIPPED: no full-SFT DCP checkpoint was produced')


### 5.4 启动转换后的 full-SFT server

它复用与 base、merged LoRA 相同的 serving 入口，但拥有独立 checkpoint 与 log identity。


In [ ]:
if RUN_FULL_SFT:
    full_sft_server_log = PATHS['logs'] / 'full_sft_server.log'
    require_fresh_paths([full_sft_server_log])
    full_sft_server_command = [
        'bash', str(TRAINING_ROOT / 'scripts/launch_official_server.sh'),
        '0', str(PORT), str(full_sft_server_log), 'False', str(FULL_SFT_MODEL),
    ]
    full_sft_server_handle = full_sft_server_log.open('xb')
    FULL_SFT_SERVER_PROCESS = subprocess.Popen(
        full_sft_server_command, cwd=ROBOTWIN, stdout=full_sft_server_handle,
        stderr=subprocess.STDOUT, start_new_session=True,
    )
    for _ in range(600):
        if FULL_SFT_SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f'Full-SFT server exited; inspect {full_sft_server_log}')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=2).read()
            print('Full-SFT server ready on port', PORT)
            break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError(f'Full-SFT server did not become ready; inspect {full_sft_server_log}')
else:
    print('SKIPPED: full-SFT server was not started')


### 5.5 评测转换后的 full-SFT checkpoint

只有 full SFT 和 conversion 完成后才运行同一 aggregate 闭环协议。这个结果不能并入 base/LoRA 的分母。


In [ ]:
if RUN_FULL_SFT:
    full_sft_eval_log = PATHS['logs'] / 'full_sft_evaluation.log'
    FULL_SFT_RESULT, FULL_SFT_ATTEMPTS, FULL_SFT_VIDEO_ARTIFACTS = collect_serial_episodes(
        'full_sft', full_sft_eval_log,
        PATHS['run_root'] / 'full_sft_attempts.json',
        PATHS['full_sft_videos'], eval_env,
    )
    print('Full-SFT aggregate result:', FULL_SFT_RESULT)
    print('Full-SFT video artifacts:', FULL_SFT_VIDEO_ARTIFACTS)
else:
    print('SKIPPED: no converted full-SFT model is being served')


### 5.6 可选 full-SFT 100 tasks × 10 episodes

这条比赛扩展默认关闭，并使用独立 benchmark run name。


In [ ]:
RUN_FULL_SFT_FULL_BENCHMARK = False
FULL_SFT_BENCHMARK_GPU_COUNT = CONFIG['competition_gpu_count']
FULL_SFT_BENCHMARK_RUN_NAME = f"{CONFIG['run_name']}-full-sft-both100x10-{FULL_SFT_BENCHMARK_GPU_COUNT}gpu"
FULL_SFT_BENCHMARK_RESUME = False
if RUN_FULL_SFT and RUN_FULL_SFT_FULL_BENCHMARK:
    if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
        FULL_SFT_SERVER_PROCESS.wait(timeout=30)
        full_sft_server_handle.close()
    full_sft_benchmark_command = [
        PYTHON, str(TRAINING_ROOT / 'scripts/run_clean_benchmark.py'),
        '--gpu-count', str(FULL_SFT_BENCHMARK_GPU_COUNT), '--episodes', '10',
        '--model-path', str(FULL_SFT_MODEL), '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', FULL_SFT_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME), '--robotwin-root', str(ROBOTWIN),
    ]
    full_sft_benchmark_root = RUNTIME / 'outputs' / FULL_SFT_BENCHMARK_RUN_NAME
    if FULL_SFT_BENCHMARK_RESUME:
        full_sft_benchmark_command.append('--resume')
    else:
        require_fresh_paths([full_sft_benchmark_root])
    subprocess.run(full_sft_benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('SKIPPED: optional full-SFT 100-task benchmark')


### 5.7 cleanup 与 artifact inventory

只停止当前 kernel 创建的 server process。最终 inventory 列出 DCP、serving checkpoint、logs 与 evidence，不自动删除它们。课程核心成果是学会运行并区分 LoRA 与全量 SFT；policy-quality 结论仍需要更强的评测证据。


In [ ]:
if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
    FULL_SFT_SERVER_PROCESS.wait(timeout=30)
    full_sft_server_handle.close()
    print('Full-SFT model server stopped')
else:
    print('No full-SFT server started by this notebook kernel')

final_inventory = {
    'run_root': str(PATHS['run_root']),
    'manifest': str(PATHS['manifest']),
    'lora_dcp': str(checkpoint) if 'checkpoint' in globals() else None,
    'merged_lora_model': str(MERGED) if 'MERGED' in globals() else None,
    'aggregate_evidence': str(PATHS['aggregate_evidence']) if PATHS['aggregate_evidence'].exists() else None,
    'base_videos': BASE_VIDEO_ARTIFACTS,
    'merged_lora_videos': MERGED_VIDEO_ARTIFACTS,
    'full_sft_dcp': str(FULL_SFT_CHECKPOINT) if RUN_FULL_SFT else None,
    'full_sft_model': str(FULL_SFT_MODEL) if RUN_FULL_SFT else None,
    'full_sft_videos': FULL_SFT_VIDEO_ARTIFACTS if RUN_FULL_SFT else None,
    'resolved_seed_matching_available': False,
}
print(json.dumps(final_inventory, indent=2))
